# 01 - Veri Kümesi Analizi (Data Analysis)

**Amaç:** Deneysel kurguda kullanılacak iki temel hizalama veri kümesinin
(`HuggingFaceH4/no_robots` ve `Anthropic/hh-rlhf`) yapısını, büyüklüğünü,
kategori/alt-küme dağılımlarını ve token uzunluk istatistiklerini incelemek.

- **Gösterim Verisi (SFT):** `HuggingFaceH4/no_robots` (Rajani vd., 2023)
- **Tercih Verisi (RLHF & DPO):** `Anthropic/hh-rlhf` (Bai vd., 2022)

Bu defter, bu depodaki `Datasets/` klasöründe önceden indirilmiş olan yerel
kopyaları kullanır ve ileride `02_SFT_Training.ipynb` → `03_RLHF_Training.ipynb`
→ `04_DPO_Training.ipynb` → `05/06_Algorithm_Comparison_*.ipynb` adımlarına
temel oluşturur.

> Bu defter Google Colab Pro (A100/L4 GPU) üzerinde çalıştırılmak üzere
> tasarlanmıştır; ancak bu aşamada sadece CPU yeterlidir (model eğitimi yoktur).

## 0. Ortam Kurulumu

In [ ]:
# --- Google Colab ortam kurulumu -------------------------------------------
# Bu hucre, projenin tamamini (Datasets/, src/, notebooks/, outputs/) iceren
# "Uygulama" klasorunu Google Drive'a yukledikten sonra calistirilmalidir.
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # TODO: Kendi Google Drive yolunuza gore guncelleyin
    PROJECT_ROOT = "/content/drive/MyDrive/Uygulama"
else:
    PROJECT_ROOT = os.path.abspath("..")

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IN_COLAB:", IN_COLAB)

In [ ]:
%pip install -q -U "transformers==4.44.2" "datasets==2.20.0" "numpy==1.26.4" sentencepiece pandas matplotlib pyarrow

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import alignment_pipeline as ap

ap.set_global_seed(42)
print("Cihaz:", ap.get_device())

In [ ]:
config = ap.ExperimentConfig(
    dataset_root=os.path.join(PROJECT_ROOT, "Datasets"),
    output_root=os.path.join(PROJECT_ROOT, "outputs"),
)
ap.ensure_dir(config.output_root)
print("Taban model:", config.base_model_id)
print("Veri kok dizini:", config.dataset_root)
print("Cikti kok dizini:", config.output_root)

## 1. Gösterim Verisi (SFT): `HuggingFaceH4/no_robots`

Bu veri kümesi, modele asistan personası kazandırmak için kullanılan,
insan tarafından yazılmış talimat–yanıt (instruction–response) çiftlerinden
oluşur ve `messages` sütununda sohbet (chat) formatındadır.

In [ ]:
no_robots = ap.load_no_robots_sft(config.dataset_root)
display(ap.dataset_overview_table(no_robots))

In [ ]:
ap.print_example(no_robots["train"][0], max_chars=400)

### 1.1 Kategori Dağılımı

In [ ]:
category_counts = ap.plot_category_bar(
    no_robots["train"], "category", "no_robots - Kategori Dağılımı (train)"
)
category_counts

### 1.2 Token Uzunluk Dağılımı

Qwen2.5-0.5B tokenizer'ı ile, sohbet şablonu (chat template) uygulanmış tam
metnin token uzunluğu dağılımı incelenir. Bu, `max_seq_length` hiperparametresinin
(SFT aşamasında) makul şekilde seçilmesine yardımcı olur.

In [ ]:
tokenizer = ap.load_tokenizer(config.base_model_id)

no_robots_with_text = ap.add_chat_text_column(no_robots["train"], tokenizer)
lengths_no_robots = ap.token_length_series(no_robots_with_text, tokenizer, "text", sample_size=2000)

ap.plot_length_histogram(
    {"no_robots (train)": lengths_no_robots},
    title="no_robots - Sohbet Metni Token Uzunluğu Dağılımı",
)

## 2. Tercih Verisi (RLHF & DPO): `Anthropic/hh-rlhf`

Dört standart alt küme (`harmless_base`, `helpful_base`, `helpful_online`,
`helpful_rejection_sampled`) birleştirilerek yüklenir. `red_team_attempts`
alt kümesi farklı bir şemaya (chosen/rejected çifti içermeyen, derecelendirilmiş
transkript) sahip olduğu için kasıtlı olarak **hariç tutulmuştur**.

In [ ]:
hh = ap.load_hh_rlhf_preference(config.dataset_root, subsets=config.hh_rlhf_subsets)
display(ap.dataset_overview_table(hh))

### 2.1 Alt Küme (Subset) Dağılımı

In [ ]:
import collections

subset_counts = dict(collections.Counter(hh["train"]["subset"]))
ap.plot_subset_bar(subset_counts, "hh-rlhf (train) - Alt Küme Dağılımı")

### 2.2 Örnek Çift: Chosen vs. Rejected

Anthropic'in orijinal verisinde bazı Unicode karakterlerin (ör. kesme işareti)
bozuk göründüğü (`�`) gözlemlenmiştir; bu, kaynak veri kümesinin bilinen bir
kalite sorunudur ve bu depodaki ayrıştırma (parsing) kodundan kaynaklanmaz.

In [ ]:
sample = hh["train"][0]
print("=== CHOSEN ===")
print(sample["chosen"][:600])
print()
print("=== REJECTED ===")
print(sample["rejected"][:600])

In [ ]:
turns = ap.split_hh_conversation(sample["chosen"])
for t in turns:
    print(f"[{t['role']}] {t['content'][:200]}")

### 2.3 Token Uzunluk Dağılımı (Chosen vs. Rejected)

Nicel değerlendirme aşamasında (bkz. `05_Algorithm_Comparison_Quantitative.ipynb`)
"uzunluk istismarı" (length exploitation) tespiti yapılacağından, kaynak veri
kümesindeki olası bir chosen/rejected uzunluk ön-yargısını (length bias) burada
erken tespit etmek faydalıdır.

In [ ]:
sample_hh = hh["train"].select(range(3000))
len_chosen = ap.token_length_series(sample_hh, tokenizer, "chosen")
len_rejected = ap.token_length_series(sample_hh, tokenizer, "rejected")

ap.plot_length_histogram(
    {"chosen": len_chosen, "rejected": len_rejected},
    title="hh-rlhf - Chosen vs. Rejected Token Uzunluğu (örneklem n=3000)",
)

print(f"Ortalama chosen uzunluğu : {len_chosen.mean():.1f} token")
print(f"Ortalama rejected uzunluğu: {len_rejected.mean():.1f} token")

## 3. Gözlemler ve Sonraki Adım

Bu defterde elde edilen gözlemler (veri büyüklükleri, kategori/alt-küme
dağılımları, token uzunlukları ve olası uzunluk ön-yargısı) ışığında:

1. SFT aşaması `02_SFT_Training.ipynb` içinde `HuggingFaceH4/no_robots` ile,
2. Ödül Modeli + PPO (RLHF) aşaması `03_RLHF_Training.ipynb` içinde,
3. DPO aşaması `04_DPO_Training.ipynb` içinde,
4. Nicel/nitel karşılaştırmalar ise `05_Algorithm_Comparison_Quantitative.ipynb`
   ve `06_Algorithm_Comparison_Qualitative.ipynb` içinde yürütülecektir.